In [13]:
# ============================================================
# Migration reconciliation demo: rule-based validation on the Hospital benchmark
# The checker only looks at the dirty table; the clean table is used ONLY for scoring.
# ============================================================
import os, re, subprocess
import numpy as np
import pandas as pd

# ---------- 1) Load the public benchmark (Raha/Baran data-cleaning project) ----------
if not os.path.exists("raha"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/BigDaMa/raha"], check=True)

dirty = pd.read_csv("raha/datasets/hospital/dirty.csv", dtype=str, keep_default_na=False)
clean = pd.read_csv("raha/datasets/hospital/clean.csv", dtype=str, keep_default_na=False)
clean.columns = dirty.columns                       # same fields in the same order, different naming style
data_cols = [c for c in dirty.columns if c != "index"]
dirty, clean = dirty[data_cols], clean[data_cols]

err = (dirty != clean)                              # ground truth: which cells are wrong
print(f"shape: {dirty.shape} | wrong cells: {int(err.values.sum())} ({err.values.mean()*100:.2f}%)")

# ---------- 2) Scoring helper ----------
def report(flags, name):
    flags = flags[data_cols]
    tp = int((flags & err).values.sum())
    fp = int((flags & ~err).values.sum())
    fn = int((~flags & err).values.sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn)
    print(f"{name:32s} flagged={tp+fp:4d}  TP={tp:3d}  FP={fp:3d}  FN={fn:3d}  precision={p:.3f}  recall={r:.3f}")
    return {"checker": name, "flagged": tp + fp, "true_errors_found": tp, "false_alarms": fp,
            "missed": fn, "precision": round(p, 3), "recall": round(r, 3)}

blank = lambda: pd.DataFrame(False, index=dirty.index, columns=data_cols)

# ---------- 3) Shortcut baseline (specific to this benchmark, NOT a real check) ----------
flag_x = dirty.apply(lambda s: s.str.contains("x", case=False))

# ---------- 4) Format rules ----------
patterns = {
    "zip": r"\d{5}", "phone": r"\d{10}", "provider_number": r"\d{5}", "state": r"[a-z]{2}",
    "score": r"\d{1,3}%|empty", "sample": r"\d+ patients|empty",
}
fmt = blank()
for c, p in patterns.items():
    fmt[c] = ~dirty[c].str.fullmatch(p)

# ---------- 5) Consistency rules (a cell that disagrees with the majority of its group) ----------
rules = [
    ("provider_number", ["name", "address_1", "address_2", "address_3", "city", "state", "zip",
                         "county", "phone", "type", "owner", "emergency_service"]),
    ("measure_code",    ["measure_name", "condition"]),
    ("measure_name",    ["measure_code", "condition"]),
    (["state", "measure_code"], ["state_average"]),
]
cons = blank()
repair = dirty.copy()                               # proposed fixes: the group majority value
for key, cols in rules:
    for c in cols:
        majority = dirty.groupby(key)[c].transform(lambda s: s.mode().iloc[0])
        bad = dirty[c] != majority
        cons[c] |= bad
        repair.loc[bad, c] = majority[bad]

# ---------- 6) Shape profiling (tested, left out of the final checker) ----------
def shape(s):
    return re.sub(r"\d+", "9", re.sub(r"[A-Za-z]+", "a", s))   # letters -> a, digits -> 9

shp = blank()
for c in data_cols:
    sh = dirty[c].map(shape)
    freq = sh.value_counts(normalize=True)
    if freq.head(3).sum() >= 0.90:                  # only columns with very uniform shapes
        shp[c] = sh.map(freq) < 0.01

# ---------- 7) Results ----------
final = fmt | cons
print()
rows = [
    report(flag_x,        "Contains 'x' (benchmark shortcut)"),
    report(fmt,           "Format rules"),
    report(cons,          "Consistency rules"),
    report(final,         "Format + consistency (final)"),
    report(shp,           "Shape profiling (not used)"),
    report(final | shp,   "All three"),
]
results = pd.DataFrame(rows)

print("\nPer-column recall of the final checker:")
print((final & err).sum().div(err.sum()).round(2).sort_values().to_string())

fixed = cons & err
print(f"\nRepairs proposed by consistency rules that restored the clean value: "
      f"{int((fixed & (repair == clean)).values.sum())} of {int(fixed.values.sum())}")

missed = err & ~final
r_, c_ = np.where(missed.values)
mx = pd.DataFrame({"column": np.array(data_cols)[c_],
                   "dirty": dirty.values[r_, c_], "clean": clean.values[r_, c_]})
print(f"\nStill missed by the final checker: {len(mx)}")
print(mx.to_string())

# ---------- 8) Save outputs for the repo ----------
results.to_csv("results.csv", index=False)
r_, c_ = np.where(final.values)
flagged = pd.DataFrame({
    "row": r_, "column": np.array(data_cols)[c_],
    "value_in_target": dirty.values[r_, c_],
    "suggested_value": repair.values[r_, c_],
    "was_actually_wrong": err.values[r_, c_],
})
flagged.loc[flagged["suggested_value"] == flagged["value_in_target"], "suggested_value"] = ""
flagged.to_csv("flagged_cells.csv", index=False)
print("\nSaved results.csv and flagged_cells.csv")

shape: (1000, 19) | wrong cells: 509 (2.68%)

Contains 'x' (benchmark shortcut) flagged= 510  TP=509  FP=  1  FN=  0  precision=0.998  recall=1.000
Format rules                     flagged= 146  TP=146  FP=  0  FN=363  precision=1.000  recall=0.287
Consistency rules                flagged= 423  TP=423  FP=  0  FN= 86  precision=1.000  recall=0.831
Format + consistency (final)     flagged= 505  TP=505  FP=  0  FN=  4  precision=1.000  recall=0.992
Shape profiling (not used)       flagged= 109  TP=100  FP=  9  FN=409  precision=0.917  recall=0.196
All three                        flagged= 515  TP=506  FP=  9  FN=  3  precision=0.983  recall=0.994

Per-column recall of the final checker:
state_average        0.93
state                0.96
type                 0.97
address_1            1.00
city                 1.00
zip                  1.00
county               1.00
provider_number      1.00
phone                1.00
owner                1.00
emergency_service    1.00
condition           